# Protein abundance change

In [4]:
# imports
import os
import polars as pl
import numpy as np
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

## Disable truncation globally
pl.Config.set_tbl_rows(20)  # Show all rows
pl.Config.set_tbl_cols(40)  # Show all columns

## define control types
TC = ["EGFP"]
NC = ["RHEB", "MAPK9", "PRKACB", "SLIRP"]
PC = ["ALK", "ALK_Arg1275Gln", "PTK2B"]
cNC = ["Renilla"]
cPC = ["KRAS", "PTK2B", "GHSR", "ABL1", "BRD4", "OPRM1", "RB1", "ADA", "WT PMP22", "LYN", "TNF", 
       "CYP2A6", "CSK", "PAK1", "ALDH2", "CHRM3", "KCNQ2", "ALK T1151M", "PRKCE", "LPAR1", "PLP1"]

%matplotlib inline

thresh = 3
min_class_num = 2

In [28]:
# Paths
metrics_dir = "../1_snakemake_pipeline/2025_varchamp_snakemake/snakemake_pipeline/outputs/analyses/{}/profiles_tcdropped_filtered_var_mad_outlier_featselect"
batches = ["2024_01_23_Batch_7", "2024_02_06_Batch_8"]

metrics_df, metrics_wtvar = pl.DataFrame(), pl.DataFrame()
for batch in batches:
    met_dir = metrics_dir.format(batch)
    metrics_df_batch = pl.read_csv(f"{met_dir}/metrics.csv")
    metrics_wtvar_batch = pl.read_csv(f"{met_dir}/metrics_summary.csv")

    metrics_df = pl.concat([metrics_df, metrics_df_batch])
    metrics_wtvar = pl.concat([metrics_wtvar, metrics_wtvar_batch])

In [ ]:
metrics_wtvar

Classifier_type,allele_0,Allele_set,Batch,AUROC_thresh,AUROC_mean,trainsize_0_mean,testsize_0_mean,trainsize_1_mean,testsize_1_mean,Training_imbalance_mean
str,str,str,str,f64,f64,f64,f64,f64,f64,f64
"""morphology""","""NEXN_Tyr544Cys""","""A1P3""","""B7""",0.997315,0.655426,970.5,323.5,963.75,321.25,1.066752
"""morphology""","""DES_Leu274Pro""","""A1P1""","""B7""",0.997315,0.909391,611.25,203.75,717.0,239.0,1.178783
"""morphology""","""PMP22_Arg157Trp""","""A1P3""","""B7""",0.997315,0.822457,1362.75,454.25,1194.75,398.25,1.149595
"""localization""","""PYGL_Asn632Ile""","""A1P3""","""B7""",0.71418,0.836791,614.25,204.75,814.5,271.5,1.324751
"""localization""","""KRT86_Ala118Glu""","""A1P2""","""B7""",0.71418,0.676701,697.5,232.5,948.75,316.25,1.374313
"""localization""","""KRT6A_Leu468Pro""","""A1P2""","""B7""",0.71418,0.905717,710.25,236.75,837.0,279.0,1.18414
"""localization""","""AGXT_Asn22Ser""","""A1P1""","""B7""",0.71418,0.64186,1403.25,467.75,1590.75,530.25,1.132169
"""morphology""","""POMT1_Ser515Arg""","""A1P3""","""B7""",0.997315,0.601311,647.25,215.75,645.0,215.0,1.077782
"""localization""","""AGXT_Arg381Lys""","""A1P1""","""B7""",0.71418,0.542241,1352.25,450.75,1590.75,530.25,1.177153


In [38]:
## get the localization metrics
local_wtvar = metrics_wtvar.filter(pl.col("Classifier_type") == "localization")

## select alleles mislocalized in both batches
local_wtvar = local_wtvar.with_columns(
    pl.when(pl.col("AUROC_mean") > pl.col("AUROC_thresh"))
    .then(1)
    .otherwise(0)
    .alias("mislocalized")
).unique()

local_wtvar = (
    local_wtvar.pivot(
        index=["allele_0", "Allele_set"],
        on="Batch",
        values="mislocalized",
    )
    .with_columns(
        ((pl.col("B7") == 1) & (pl.col("B8") == 1)).alias(
            "mislocalized_both_batches"
        )
    )
    .rename({"B7": "mislocalized_batch7", "B8": "mislocalized_batch8"})
)

misloc_alleles = (
    local_wtvar
    # .filter(pl.col("Mislocalized_both_batches"))
    .select("allele_0")
    .to_series()
    .to_list()
)

misloc_wt = (
    local_wtvar
    # .filter(pl.col("Mislocalized_both_batches"))
    .with_columns(pl.col("allele_0").str.replace("_.*", "").alias("wt"))
    .select("wt")
    .to_series()
    .unique()
    .to_list()
)

all_alleles = misloc_alleles + misloc_wt

# Get individual classifiers
ind_class = metrics_df.filter(
    (pl.col("Metadata_Control")!="true")
    & (pl.col("Training_imbalance") < thresh)
    & (
        (pl.col("Full_Classifier_ID").str.contains("true"))
        & pl.col("allele_0").is_in(misloc_alleles)
    )
)

In [39]:
ind_class

AUROC,AUPRC,AUBPRC,Macro_F1,Sensitivity,Specificity,Balanced_Accuracy,Classifier_ID,Full_Classifier_ID,Plate,trainsize_0,testsize_0,well_0,allele_0,trainsize_1,testsize_1,well_1,allele_1,Metadata_Control,train_prob_1,test_prob_1,Training_imbalance,Testing_imbalance
f64,f64,f64,f64,f64,f64,f64,str,str,str,i64,i64,str,str,i64,i64,str,str,bool,f64,f64,f64,f64
0.600219,0.723602,0.587594,0.523385,0.463807,0.640394,0.552101,"""2024_01_22_B7A1R1_P4T4_H09_F09""","""2024_01_22_B7A1R1_P4T4_H09_F09…","""2024_01_22_B7A1R1_P4T4""",1153,203,"""H09""","""TTPA_Asp64Gly""",1070,373,"""F09""","""TTPA""",false,0.481332,0.647569,1.07757,1.837438
0.707959,0.723445,0.748561,0.654405,0.493724,0.823529,0.658627,"""2024_01_17_B7A1R1_P1T3_M02_K02""","""2024_01_17_B7A1R1_P1T3_M02_K02…","""2024_01_17_B7A1R1_P1T3""",702,272,"""M02""","""AMPD2_Glu697Asp""",579,239,"""K02""","""AMPD2""",false,0.451991,0.46771,1.212435,1.138075
0.83739,0.870089,0.85311,0.751481,0.842995,0.657382,0.750188,"""2024_01_17_B7A1R1_P2T4_C08_M06""","""2024_01_17_B7A1R1_P2T4_C08_M06…","""2024_01_17_B7A1R1_P2T4""",670,359,"""C08""","""GSS_Arg283Cys""",1074,414,"""M06""","""GSS""",false,0.615826,0.535576,1.602985,1.153203
0.628188,0.437454,0.613327,0.573296,0.357616,0.785714,0.571665,"""2024_01_19_B7A1R1_P3T4_D20_P18""","""2024_01_19_B7A1R1_P3T4_D20_P18…","""2024_01_19_B7A1R1_P3T4""",1782,616,"""D20""","""SDC3_Val136Ile""",1170,302,"""P18""","""SDC3""",false,0.396341,0.328976,1.523077,2.039735
0.766978,0.66026,0.804085,0.689803,0.522034,0.844302,0.683168,"""2024_01_17_B7A1R1_P1T3_B12_J21""","""2024_01_17_B7A1R1_P1T3_B12_J21…","""2024_01_17_B7A1R1_P1T3""",1494,623,"""B12""","""DES_Pro419Ser""",661,295,"""J21""","""DES""",false,0.306729,0.321351,2.260212,2.111864
0.796155,0.84983,0.814184,0.709856,0.643341,0.795918,0.71963,"""2024_01_17_B7A1R1_P1T1_E11_E07""","""2024_01_17_B7A1R1_P1T1_E11_E07…","""2024_01_17_B7A1R1_P1T1""",1522,343,"""E11""","""AGXT_Gly82Arg""",1678,443,"""E07""","""AGXT""",false,0.524375,0.563613,1.102497,1.291545
0.97593,0.990753,0.97557,0.922859,0.972727,0.853659,0.913193,"""2024_01_17_B7A1R1_P1T2_M03_A01""","""2024_01_17_B7A1R1_P1T2_M03_A01…","""2024_01_17_B7A1R1_P1T2""",299,123,"""M03""","""ACSF3_Met266Val""",745,330,"""A01""","""ACSF3""",false,0.713602,0.728477,2.491639,2.682927
0.490225,0.510048,0.512524,0.497601,0.543825,0.453649,0.498737,"""2024_01_17_B7A1R1_P1T3_K19_E07""","""2024_01_17_B7A1R1_P1T3_K19_E07…","""2024_01_17_B7A1R1_P1T3""",1179,507,"""K19""","""AGXT_Arg197Gln""",1619,502,"""E07""","""AGXT""",false,0.578628,0.497522,1.373198,1.00996
0.623357,0.757128,0.659568,0.580232,0.607477,0.56391,0.585693,"""2024_01_17_B7A1R1_P1T1_B06_J21""","""2024_01_17_B7A1R1_P1T1_B06_J21…","""2024_01_17_B7A1R1_P1T1""",566,133,"""B06""","""DES_Asp343Asn""",742,214,"""J21""","""DES""",false,0.567278,0.616715,1.310954,1.609023


## Filter the cells passed QC

In [40]:
pass_qc_prof_dir = "../1_snakemake_pipeline/2025_varchamp_snakemake/snakemake_pipeline/outputs/batch_profiles/{}/profiles_tcdropped_filtered_var_mad_outlier_featselect.parquet"

# Get meta features
b7_alleles = (
    pl.scan_parquet(
        pass_qc_prof_dir.format(batches[0])
    )
    .filter(pl.col("Metadata_gene_allele").is_in(all_alleles))
    .with_columns(
        pl.concat_str(
            [
                "Metadata_Plate",
                "Metadata_Well",
                "Metadata_ImageNumber",
                "Metadata_ObjectNumber",
            ],
            separator="_",
        ).alias("Metadata_CellID")
    )
    .select([
        "Metadata_CellID",
        "Metadata_gene_allele",
        "Metadata_Well",
        "Metadata_Plate",
    ])
)

b8_alleles = (
    pl.scan_parquet(
        pass_qc_prof_dir.format(batches[1])
    )
    .filter(pl.col("Metadata_gene_allele").is_in(all_alleles))
    .with_columns(
        pl.concat_str(
            [
                "Metadata_Plate",
                "Metadata_Well",
                "Metadata_ImageNumber",
                "Metadata_ObjectNumber",
            ],
            separator="_",
        ).alias("Metadata_CellID")
    )
    .select([
        "Metadata_CellID",
        "Metadata_gene_allele",
        "Metadata_Well",
        "Metadata_Plate",
    ])
)

cell_alleles = pl.concat([b7_alleles.collect(), b8_alleles.collect()])

In [41]:
prof_dir = "../1_snakemake_pipeline/2025_varchamp_snakemake/snakemake_pipeline/outputs/batch_profiles/{}/profiles.parquet"

b7_prof = pl.scan_parquet(
    prof_dir.format(batches[0])
).with_columns(
    pl.concat_str(
        [
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_ImageNumber",
            "Metadata_ObjectNumber",
        ],
        separator="_",
    ).alias("Metadata_CellID")
)
b8_prof = pl.scan_parquet(
    prof_dir.format(batches[1])
).with_columns(
    pl.concat_str(
        [
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_ImageNumber",
            "Metadata_ObjectNumber",
        ],
        separator="_",
    ).alias("Metadata_CellID")
)

gfp_int = [i for i in b7_prof.collect_schema().names() if "Cells_Intensity" in i]
gfp_int = ["Metadata_CellID"] + [i for i in gfp_int if "GFP" in i]

profiles = pl.concat([
    b7_prof.select(gfp_int).collect(),
    b8_prof.select(gfp_int).collect(),
])

# This method of joining ensures that only cells past filters are included in the abundance analysis
profiles = cell_alleles.join(profiles, on="Metadata_CellID", how="left")

In [42]:
well_profiles = (
    profiles.group_by(["Metadata_Plate", "Metadata_Well", "Metadata_gene_allele"])
    .agg(
        pl.col(col).median().alias(col)
        for col in profiles.columns
        if not col.startswith("Metadata_")
    )
    # .drop("Metadata_CellID")
    .unique()
)

well_profiles

Metadata_Plate,Metadata_Well,Metadata_gene_allele,Cells_Intensity_IntegratedIntensityEdge_GFP,Cells_Intensity_IntegratedIntensity_GFP,Cells_Intensity_LowerQuartileIntensity_GFP,Cells_Intensity_MADIntensity_GFP,Cells_Intensity_MassDisplacement_GFP,Cells_Intensity_MaxIntensityEdge_GFP,Cells_Intensity_MaxIntensity_GFP,Cells_Intensity_MeanIntensityEdge_GFP,Cells_Intensity_MeanIntensity_GFP,Cells_Intensity_MedianIntensity_GFP,Cells_Intensity_MinIntensityEdge_GFP,Cells_Intensity_MinIntensity_GFP,Cells_Intensity_StdIntensityEdge_GFP,Cells_Intensity_StdIntensity_GFP,Cells_Intensity_UpperQuartileIntensity_GFP
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2024_02_05_B8A1R2_P4T4""","""H15""","""UNC119""",0.667243,12.993305,0.001792,0.000108,0.417819,0.002541,0.003421,0.001811,0.001902,0.001888,0.001526,0.001465,0.000141,0.000175,0.002004
"""2024_01_19_B7A1R1_P3T2""","""F21""","""PMP22_Ala135Thr""",9.532847,266.033488,0.026912,0.00424,2.709426,0.09095,0.19126,0.025775,0.035793,0.031259,0.004321,0.003854,0.014917,0.015395,0.038936
"""2024_02_05_B8A1R2_P4T2""","""D11""","""TTPA_Arg192His""",6.542323,248.075471,0.02787,0.014334,3.057163,0.061906,0.193519,0.016006,0.047851,0.042396,0.002709,0.002614,0.010322,0.027498,0.058438
"""2024_02_01_B8A1R2_P2T2""","""O15""","""GCK_Val253Ala""",1.22839,37.86619,0.003382,0.001096,2.987357,0.007292,0.010583,0.002684,0.00466,0.004693,0.001599,0.001594,0.001139,0.001728,0.005766
"""2024_01_31_B8A1R2_P1T4""","""P01""","""COMP_Ala171Thr""",0.786815,15.598635,0.001618,0.000078,0.461515,0.002001,0.002197,0.001643,0.001696,0.001692,0.001329,0.001258,0.000113,0.000118,0.00177
"""2024_01_17_B7A1R1_P2T3""","""A14""","""HBD_Asn13Lys""",0.975927,20.106093,0.002135,0.000344,1.939039,0.004121,0.005744,0.002175,0.002661,0.002536,0.001545,0.001523,0.000446,0.000687,0.00297
"""2024_01_18_B7A1R1_P3T1""","""P07""","""PKP2_Met365Ile""",1.155704,24.139853,0.003263,0.000363,1.699114,0.009835,0.015742,0.002947,0.003916,0.003678,0.001608,0.001602,0.00118,0.001225,0.004086
"""2024_01_17_B7A1R1_P2T2""","""P19""","""KCTD7_Arg184Cys""",1.206669,32.446731,0.003512,0.001205,3.263968,0.008008,0.018544,0.003032,0.005168,0.004685,0.00159,0.001575,0.001439,0.002184,0.006082
"""2024_01_31_B8A1R2_P1T2""","""P17""","""DCX_Ile214Thr""",1.869048,73.781299,0.005533,0.002385,3.200446,0.015214,0.02569,0.003766,0.008213,0.007735,0.00163,0.00163,0.002374,0.003564,0.010423


## Calculate abundance hits

In [43]:
import statsmodels.formula.api as smf
from scipy.stats import chi2
from statsmodels.stats.anova import anova_lm

def lmm(dat, fix_var: str, reference: str, var: str, rand_var: str):
    ## basic test
    model = smf.ols(f"{fix_var} ~ C(Metadata_gene_allele, Treatment(reference='{reference}'))", 
                    data=dat)

    ### Also tried a fixed effect model, no improvement, keep it simple
    # fixed_model = smf.ols(f"{fix_var} ~ C(Metadata_gene_allele, Treatment(reference='{reference}')) + C({rand_var})", 
    #                       data=dat).fit()
    ### Not informative due to the large cell counts
    # anova_results = anova_lm(model, fixed_model)
    # print(anova_results)

    ### No significant difference between adding or removing the variable correction
    # print("Fixed model R²:", fixed_model.rsquared_adj)
    # print("Non-fixed model R²:", model.rsquared_adj)
    
    result = model.fit()
    # Extract the summary table for Logit
    summary_table = result.summary2().tables[1]
    # print(pl.DataFrame(summary_table))

    ### mixed LM: ConvergenceWarning: The MLE may be on the boundary of the parameter space. Disgarded
    # model = smf.mixedlm(
    #     formula=f"{fix_var} ~ C(Metadata_gene_allele, Treatment(reference='{reference}'))", ##  + C({rand_var})
    #     data=dat,
    #     groups=dat[rand_var],
    #     re_formula="1" ## Only preserve random intercept, not slope
    # )
    # result = model.fit() ## method='powell' | maxiter=1000, method="bfgs"
    ### Extract the summary table for Logit
    # summary_table = result.summary().tables[1]

    ### format results
    summary_df = pl.DataFrame(summary_table[1:2])
    summary_df = summary_df.with_columns([
        pl.col(column).cast(pl.Float64).alias(column) for column in summary_df.columns
    ])
    summary_df = summary_df.with_columns(
        pl.lit(reference).alias("Gene"), pl.lit(var).alias("Variant")
    )

    return summary_df

In [44]:
from scipy.stats import shapiro
import re
from scipy.stats import wilcoxon
from scipy.stats import ttest_rel
import pandas as pd


# Convert letter rows to numbers
def well_to_coordinates(well):
    row_letter, col_number = re.match(r"([A-P])(\d{2})", well).groups()
    row_index = ord(row_letter) - ord('A') + 1  # Convert 'A'->1, 'B'->2, ..., 'P'->16
    col_index = int(col_number)  # Convert string column to integer
    return well, row_index, col_index


# Compute distances from edges and find the most centered well
def compute_distance(row, col):
    return min(row - 1, 16 - row, col - 1, 24 - col)  # Distance from nearest edge
    

def paired_ttest(dat, reference: str, var: str):
    ## perform stat test
    dat = dat.dropna().sort_values(["Metadata_Plate","Metadata_gene_allele"])
    t_stat, p_val = ttest_rel(dat[dat["Metadata_gene_allele"]==reference]["Cells_Intensity_MedianIntensity_GFP"].values, 
                              dat[dat["Metadata_gene_allele"]==var]["Cells_Intensity_MedianIntensity_GFP"].values)
    summary_df = pl.DataFrame(
        {
            "t_stat": t_stat,
            "p_val": p_val
        }
    )
    summary_df = summary_df.with_columns(
        pl.lit(reference).alias("Gene"), pl.lit(var).alias("Variant")
    )
    return summary_df


well_abun_stats = []
for allele in tqdm(misloc_alleles):
    temp_class = ind_class.filter(pl.col("allele_0") == allele)
    temp_prof = well_profiles.filter(
        (pl.col("Metadata_Plate").is_in(temp_class.select("Plate").to_series()))
        & (
            pl.col("Metadata_Well").is_in(temp_class.select("well_1"))
            | pl.col("Metadata_Well").is_in(temp_class.select("well_0"))
        )
    )
    reference = allele.split("_")[0]

    temp_prof = temp_prof.to_pandas().dropna().sort_values(["Metadata_Plate","Metadata_gene_allele"])
    var_profiles = temp_prof[temp_prof["Metadata_gene_allele"]==allele]
    ref_profiles = temp_prof[temp_prof["Metadata_gene_allele"]==reference]
    
    # print(temp_prof)
    # diff = temp_prof[temp_prof["Metadata_gene_allele"]==allele]["Cells_Intensity_IntegratedIntensityEdge_GFP"].values - \
    #        temp_prof[temp_prof["Metadata_gene_allele"]==reference]["Cells_Intensity_IntegratedIntensityEdge_GFP"].values

    # try:
    if (ref_profiles.shape[0] > var_profiles.shape[0]):
        ref_al_wells = ref_profiles["Metadata_Well"].unique()
        well_coords = [well_to_coordinates(w) for w in set([ref_well_pl for ref_well_pl in ref_al_wells])]
        # Sort wells by max distance from edges (descending)
        most_centered_well = max(well_coords, key=lambda x: compute_distance(x[1], x[2]))[0]
        ref_profiles = temp_prof[(temp_prof["Metadata_gene_allele"]==reference)&(temp_prof["Metadata_Well"]==most_centered_well)]
        
    if (ref_profiles.shape[0] < var_profiles.shape[0]):
        ref_al_wells = var_profiles["Metadata_Well"].unique()
        well_coords = [well_to_coordinates(w) for w in set([ref_well_pl for ref_well_pl in ref_al_wells])]
        # Sort wells by max distance from edges (descending)
        most_centered_well = max(well_coords, key=lambda x: compute_distance(x[1], x[2]))[0]
        var_profiles = temp_prof[(temp_prof["Metadata_gene_allele"]==allele)&(temp_prof["Metadata_Well"]==most_centered_well)]

    well_abun_stats.append(
        paired_ttest(
            dat=pd.concat([ref_profiles, var_profiles]),
            reference=reference,
            var=allele
        )
    )

well_abun_stats = pl.concat(well_abun_stats, how="vertical")
well_abun_stats = well_abun_stats.rename({"t_stat": "U2OS_t"})
well_abun_stats = well_abun_stats.sort(["Gene", "Variant", "U2OS_t", "p_val"])
well_abun_stats

100%|██████████| 610/610 [00:25<00:00, 23.49it/s]


U2OS_t,p_val,Gene,Variant
f64,f64,str,str
-1.045202,0.372731,"""ACACB""","""ACACB_Val958Met"""
7.045782,0.005876,"""ACSF3""","""ACSF3_Ala197Thr"""
3.177852,0.050177,"""ACSF3""","""ACSF3_Arg10Trp"""
4.021488,0.00505,"""ACSF3""","""ACSF3_Arg471Trp"""
3.393741,0.01154,"""ACSF3""","""ACSF3_Arg558Trp"""
3.888603,0.005986,"""ACSF3""","""ACSF3_Asp457Asn"""
8.580732,0.003327,"""ACSF3""","""ACSF3_Glu359Lys"""
2.012066,0.090894,"""ACSF3""","""ACSF3_Gly119Asp"""
6.571809,0.007167,"""ACSF3""","""ACSF3_Gly225Arg"""


In [46]:
well_abun_stats.write_csv("../../3_outputs/abundance/abundance_ttest_stats.csv")